In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

SUPP_DIR = Path("D:/Lumiere/data sheet 1")

print("Working directory:")
print(SUPP_DIR)

print("Exists:", SUPP_DIR.exists())

Working directory:
C:\Lumiere\data sheet 1
Exists: True


In [2]:
pathway_file = SUPP_DIR / "Supplementary Table S10.XLSX"

pathway = {}

for sheet in ["Drought", "Heat", "Drought Vs Heat"]:
    pathway[sheet] = pd.read_excel(
        pathway_file,
        sheet_name=sheet
    )

    print(sheet, pathway[sheet].shape)

Drought (11, 18)
Heat (1, 19)
Drought Vs Heat (9, 19)


In [3]:
pathway_all = pd.concat(
    [
        pathway["Drought"].assign(Condition="Drought"),
        pathway["Heat"].assign(Condition="Heat"),
        pathway["Drought Vs Heat"].assign(Condition="Drought_vs_Heat")
    ],
    ignore_index=True
)

print(pathway_all.shape)

pathway_all.head()

(21, 20)


,Entry,protein_id,Gene_ID,baseMean,log2FoldChange,lfcSE,pvalue,padj,pident,evalue,bitscore,Reviewed,Protein.names,Gene.Names,Organism,GO_ID,Protein.families,KEGG,Condition,Unnamed: 0
0,A0A191T7N6,KAF9605401.1,IFM89_016978,1066.087286,6.402448,1.364008,1.870000e-07,0.000009,41.9,2.000000e-43,149.0,unreviewed,S-norcoclaurine synthase (EC 4.2.1.78) (S-norc...,LOC104590414 HUJ06_019302,Nelumbo nucifera (Sacred lotus),GO:0004864; GO:0005634; GO:0005737; GO:0006952...,NaN,nnu:104590414;,Drought,NaN
1,A0A191T7N6,KAF9607136.1,IFM89_032358,151.792503,3.933221,1.496672,3.924980e-04,0.005290,47.4,6.840000e-47,157.0,unreviewed,S-norcoclaurine synthase (EC 4.2.1.78) (S-norc...,LOC104590414 HUJ06_019302,Nelumbo nucifera (Sacred lotus),GO:0004864; GO:0005634; GO:0005737; GO:0006952...,NaN,nnu:104590414;,Drought,NaN
2,A0A0N7F294,KAF9599036.1,IFM89_033381,1043.210917,-1.500283,0.521687,8.003250e-04,0.009408,81.9,1.230000e-215,599.0,unreviewed,O-methyltransferase,OMT4,Sinopodophyllum hexandrum (Himalayan may apple...,GO:0008171; GO:0032259,NaN,NaN,Drought,NaN
3,A0A1U7ZQ12,KAF9606077.1,IFM89_023100,409.877091,-1.673854,0.678748,2.300796e-03,0.021880,44.2,3.390000e-124,375.0,unreviewed,(S)-N-methylcoclaurine 3'-hydroxylase isozyme ...,LOC104595105 CYP80B2,Nelumbo nucifera (Sacred lotus),GO:0004497; GO:0005506; GO:0016705; GO:0020037,Cytochrome P450 family,nnu:104595105;,Drought,NaN
4,A0A097H1C2,KAF9595526.1,IFM89_000614,491.925073,-1.750780,0.494353,6.720000e-05,0.001260,56.6,2.650000e-64,203.0,unreviewed,Norcoclaurine synthase (EC 4.2.1.78),NCS,Sinopodophyllum hexandrum (Himalayan may apple...,GO:0006952; GO:0050474,NaN,NaN,Drought,NaN


In [4]:
# Remove accidental Excel index columns

pathway_all = pathway_all.drop(
    columns=[
        col for col in pathway_all.columns
        if col.startswith("Unnamed")
    ],
    errors="ignore"
)

# Clean gene names
pathway_all["Gene.Names"] = (
    pathway_all["Gene.Names"]
    .fillna("")
    .astype(str)
    .str.strip()
)

pathway_all["Protein.names"] = (
    pathway_all["Protein.names"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print("Shape:", pathway_all.shape)

display(pathway_all[
    [
        "Gene.Names",
        "Protein.names",
        "log2FoldChange",
        "padj",
        "Condition"
    ]
])

Shape: (21, 19)


,Gene.Names,Protein.names,log2FoldChange,padj,Condition
0,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norc...,6.402448,8.620000e-06,Drought
1,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norc...,3.933221,5.289941e-03,Drought
2,OMT4,O-methyltransferase,-1.500283,9.408498e-03,Drought
3,LOC104595105 CYP80B2,(S)-N-methylcoclaurine 3'-hydroxylase isozyme ...,-1.673854,2.188026e-02,Drought
4,NCS,Norcoclaurine synthase (EC 4.2.1.78),-1.750780,1.259917e-03,Drought
5,,Berberine bridge enzyme (EC 1.21.3.3),-2.015363,1.469993e-03,Drought
6,OMT2,S-adenosyl-L-methionine:O-methyltransferase,-2.419962,3.396439e-02,Drought
7,HNMC4OMT,3'-hydroxy-N-methyl-(S)-coclaurine 4'-O-methyl...,-3.462434,3.980000e-08,Drought
8,OMT6,S-adenosyl-L-methionine:O-methyltransferase,-4.538413,1.300000e-09,Drought
9,NC6OMT,Norcoclaurine 6-O-methyltransferase (EC 2.1.1....,-6.677537,4.340000e-06,Drought


In [5]:
candidate_table = pathway_all[
    [
        "Gene.Names",
        "Protein.names",
        "log2FoldChange",
        "padj",
        "Condition",
        "GO_ID",
        "KEGG"
    ]
].copy()


candidate_table = candidate_table.sort_values(
    by="log2FoldChange",
    ascending=False
)


print("Candidate genes:", len(candidate_table))

display(candidate_table)

Candidate genes: 21


,Gene.Names,Protein.names,log2FoldChange,padj,Condition,GO_ID,KEGG
0,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norc...,6.402448,8.620000e-06,Drought,GO:0004864; GO:0005634; GO:0005737; GO:0006952...,nnu:104590414;
1,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norc...,3.933221,5.289941e-03,Drought,GO:0004864; GO:0005634; GO:0005737; GO:0006952...,nnu:104590414;
12,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norc...,2.842387,2.254623e-02,Drought_vs_Heat,GO:0004864; GO:0005634; GO:0005737; GO:0006952...,nnu:104590414;
13,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norc...,2.587895,3.361919e-02,Drought_vs_Heat,GO:0004864; GO:0005634; GO:0005737; GO:0006952...,nnu:104590414;
14,NCS,Norcoclaurine synthase (EC 4.2.1.78),-1.168098,3.629229e-02,Drought_vs_Heat,GO:0006952; GO:0050474,NaN
2,OMT4,O-methyltransferase,-1.500283,9.408498e-03,Drought,GO:0008171; GO:0032259,NaN
3,LOC104595105 CYP80B2,(S)-N-methylcoclaurine 3'-hydroxylase isozyme ...,-1.673854,2.188026e-02,Drought,GO:0004497; GO:0005506; GO:0016705; GO:0020037,nnu:104595105;
4,NCS,Norcoclaurine synthase (EC 4.2.1.78),-1.750780,1.259917e-03,Drought,GO:0006952; GO:0050474,NaN
5,,Berberine bridge enzyme (EC 1.21.3.3),-2.015363,1.469993e-03,Drought,GO:0050468; GO:0071949,NaN
15,D0Y65_033085 glysoja_035139,S-norcoclaurine synthase (S-norcoclaurine synt...,-2.245025,1.520581e-02,Drought_vs_Heat,GO:0006952,NaN


In [6]:
RESULT_DIR = Path("D:/Lumiere/Project/results")

RESULT_DIR.mkdir(
    exist_ok=True
)

candidate_table.to_csv(
    RESULT_DIR / "pathway_candidate_genes.csv",
    index=False
)

print("Saved:")
print(RESULT_DIR / "pathway_candidate_genes.csv")


Saved:
C:\Lumiere\Project\results\pathway_candidate_genes.csv


In [7]:
print("Rows:", candidate_table.shape[0])
print("Columns:", candidate_table.shape[1])

display(candidate_table)

Rows: 21
Columns: 7


,Gene.Names,Protein.names,log2FoldChange,padj,Condition,GO_ID,KEGG
0,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norc...,6.402448,8.620000e-06,Drought,GO:0004864; GO:0005634; GO:0005737; GO:0006952...,nnu:104590414;
1,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norc...,3.933221,5.289941e-03,Drought,GO:0004864; GO:0005634; GO:0005737; GO:0006952...,nnu:104590414;
12,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norc...,2.842387,2.254623e-02,Drought_vs_Heat,GO:0004864; GO:0005634; GO:0005737; GO:0006952...,nnu:104590414;
13,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norc...,2.587895,3.361919e-02,Drought_vs_Heat,GO:0004864; GO:0005634; GO:0005737; GO:0006952...,nnu:104590414;
14,NCS,Norcoclaurine synthase (EC 4.2.1.78),-1.168098,3.629229e-02,Drought_vs_Heat,GO:0006952; GO:0050474,NaN
2,OMT4,O-methyltransferase,-1.500283,9.408498e-03,Drought,GO:0008171; GO:0032259,NaN
3,LOC104595105 CYP80B2,(S)-N-methylcoclaurine 3'-hydroxylase isozyme ...,-1.673854,2.188026e-02,Drought,GO:0004497; GO:0005506; GO:0016705; GO:0020037,nnu:104595105;
4,NCS,Norcoclaurine synthase (EC 4.2.1.78),-1.750780,1.259917e-03,Drought,GO:0006952; GO:0050474,NaN
5,,Berberine bridge enzyme (EC 1.21.3.3),-2.015363,1.469993e-03,Drought,GO:0050468; GO:0071949,NaN
15,D0Y65_033085 glysoja_035139,S-norcoclaurine synthase (S-norcoclaurine synt...,-2.245025,1.520581e-02,Drought_vs_Heat,GO:0006952,NaN


In [8]:
upregulated = candidate_table[
    candidate_table["log2FoldChange"] > 0
].copy()

downregulated = candidate_table[
    candidate_table["log2FoldChange"] < 0
].copy()

print("Upregulated:")
print(upregulated.shape)

display(upregulated)

print("\nDownregulated:")
print(downregulated.shape)

display(downregulated)

Upregulated:
(4, 7)


,Gene.Names,Protein.names,log2FoldChange,padj,Condition,GO_ID,KEGG
0,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norc...,6.402448,0.000009,Drought,GO:0004864; GO:0005634; GO:0005737; GO:0006952...,nnu:104590414;
1,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norc...,3.933221,0.005290,Drought,GO:0004864; GO:0005634; GO:0005737; GO:0006952...,nnu:104590414;
12,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norc...,2.842387,0.022546,Drought_vs_Heat,GO:0004864; GO:0005634; GO:0005737; GO:0006952...,nnu:104590414;
13,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norc...,2.587895,0.033619,Drought_vs_Heat,GO:0004864; GO:0005634; GO:0005737; GO:0006952...,nnu:104590414;



Downregulated:
(17, 7)


,Gene.Names,Protein.names,log2FoldChange,padj,Condition,GO_ID,KEGG
14,NCS,Norcoclaurine synthase (EC 4.2.1.78),-1.168098,3.629229e-02,Drought_vs_Heat,GO:0006952; GO:0050474,NaN
2,OMT4,O-methyltransferase,-1.500283,9.408498e-03,Drought,GO:0008171; GO:0032259,NaN
3,LOC104595105 CYP80B2,(S)-N-methylcoclaurine 3'-hydroxylase isozyme ...,-1.673854,2.188026e-02,Drought,GO:0004497; GO:0005506; GO:0016705; GO:0020037,nnu:104595105;
4,NCS,Norcoclaurine synthase (EC 4.2.1.78),-1.750780,1.259917e-03,Drought,GO:0006952; GO:0050474,NaN
5,,Berberine bridge enzyme (EC 1.21.3.3),-2.015363,1.469993e-03,Drought,GO:0050468; GO:0071949,NaN
15,D0Y65_033085 glysoja_035139,S-norcoclaurine synthase (S-norcoclaurine synt...,-2.245025,1.520581e-02,Drought_vs_Heat,GO:0006952,NaN
11,,Stylopine synthase,-2.247584,1.963359e-02,Heat,GO:0004497; GO:0005506; GO:0016705; GO:0020037...,NaN
6,OMT2,S-adenosyl-L-methionine:O-methyltransferase,-2.419962,3.396439e-02,Drought,GO:0008171; GO:0009820; GO:0032259; GO:0046983,NaN
16,,Berberine bridge enzyme (EC 1.21.3.3),-2.511092,2.210000e-05,Drought_vs_Heat,GO:0050468; GO:0071949,NaN
17,D0Y65_033085 glysoja_035139,S-norcoclaurine synthase (S-norcoclaurine synt...,-2.860257,4.516094e-03,Drought_vs_Heat,GO:0006952,NaN


In [9]:
keywords = [
    "norcoclaurine",
    "methyltransferase",
    "oxygenase",
    "cytochrome",
    "berberine",
    "bridge",
    "oxidoreductase"
]

mask = pathway_all["Protein.names"].str.contains(
    "|".join(keywords),
    case=False,
    na=False
)

pathway_core = pathway_all[mask]

print("Core pathway candidates:")
print(pathway_core.shape)

display(
    pathway_core[
        [
            "Gene.Names",
            "Protein.names",
            "log2FoldChange",
            "padj",
            "Condition"
        ]
    ]
)

Core pathway candidates:
(18, 19)


,Gene.Names,Protein.names,log2FoldChange,padj,Condition
0,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norc...,6.402448,8.620000e-06,Drought
1,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norc...,3.933221,5.289941e-03,Drought
2,OMT4,O-methyltransferase,-1.500283,9.408498e-03,Drought
4,NCS,Norcoclaurine synthase (EC 4.2.1.78),-1.750780,1.259917e-03,Drought
5,,Berberine bridge enzyme (EC 1.21.3.3),-2.015363,1.469993e-03,Drought
6,OMT2,S-adenosyl-L-methionine:O-methyltransferase,-2.419962,3.396439e-02,Drought
7,HNMC4OMT,3'-hydroxy-N-methyl-(S)-coclaurine 4'-O-methyl...,-3.462434,3.980000e-08,Drought
8,OMT6,S-adenosyl-L-methionine:O-methyltransferase,-4.538413,1.300000e-09,Drought
9,NC6OMT,Norcoclaurine 6-O-methyltransferase (EC 2.1.1....,-6.677537,4.340000e-06,Drought
10,LOC104585771,caffeic acid 3-O-methyltransferase 1,-9.442301,3.020000e-06,Drought


In [10]:
pd.set_option(
    "display.max_colwidth",
    80
)

display(
    candidate_table[
        [
            "Gene.Names",
            "Protein.names",
            "log2FoldChange",
            "padj",
            "Condition"
        ]
    ].sort_values(
        "log2FoldChange",
        ascending=False
    )
)

,Gene.Names,Protein.names,log2FoldChange,padj,Condition
0,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norcoclaurine synthase 1-like),6.402448,8.620000e-06,Drought
1,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norcoclaurine synthase 1-like),3.933221,5.289941e-03,Drought
12,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norcoclaurine synthase 1-like),2.842387,2.254623e-02,Drought_vs_Heat
13,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norcoclaurine synthase 1-like),2.587895,3.361919e-02,Drought_vs_Heat
14,NCS,Norcoclaurine synthase (EC 4.2.1.78),-1.168098,3.629229e-02,Drought_vs_Heat
2,OMT4,O-methyltransferase,-1.500283,9.408498e-03,Drought
3,LOC104595105 CYP80B2,(S)-N-methylcoclaurine 3'-hydroxylase isozyme 2 (probable (S)-N-methylcoclau...,-1.673854,2.188026e-02,Drought
4,NCS,Norcoclaurine synthase (EC 4.2.1.78),-1.750780,1.259917e-03,Drought
5,,Berberine bridge enzyme (EC 1.21.3.3),-2.015363,1.469993e-03,Drought
15,D0Y65_033085 glysoja_035139,S-norcoclaurine synthase (S-norcoclaurine synthase 2),-2.245025,1.520581e-02,Drought_vs_Heat


In [11]:
# Create gene-level summary

gene_level = (
    candidate_table
    .groupby(
        [
            "Gene.Names",
            "Protein.names",
            "Condition"
        ],
        dropna=False
    )
    .agg(
        max_log2FC=("log2FoldChange", "max"),
        min_log2FC=("log2FoldChange", "min"),
        best_padj=("padj", "min")
    )
    .reset_index()
)

print("Gene-level entries:", gene_level.shape)

display(gene_level)

Gene-level entries: (18, 6)


,Gene.Names,Protein.names,Condition,max_log2FC,min_log2FC,best_padj
0,,Berberine bridge enzyme (EC 1.21.3.3),Drought,-2.015363,-2.015363,1.469993e-03
1,,Berberine bridge enzyme (EC 1.21.3.3),Drought_vs_Heat,-2.511092,-2.511092,2.210000e-05
2,,Stylopine synthase,Drought_vs_Heat,-4.279584,-4.279584,2.820000e-06
3,,Stylopine synthase,Heat,-2.247584,-2.247584,1.963359e-02
4,D0Y65_033085 glysoja_035139,S-norcoclaurine synthase (S-norcoclaurine synthase 2),Drought_vs_Heat,-2.245025,-2.860257,4.516094e-03
5,HNMC4OMT,3'-hydroxy-N-methyl-(S)-coclaurine 4'-O-methyltransferase (EC 2.1.1.116),Drought,-3.462434,-3.462434,3.980000e-08
6,LOC104585771,caffeic acid 3-O-methyltransferase 1,Drought,-9.442301,-9.442301,3.020000e-06
7,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norcoclaurine synthase 1-like),Drought,6.402448,3.933221,8.620000e-06
8,LOC104590414 HUJ06_019302,S-norcoclaurine synthase (EC 4.2.1.78) (S-norcoclaurine synthase 1-like),Drought_vs_Heat,2.842387,2.587895,2.254623e-02
9,LOC104595105 CYP80B2,(S)-N-methylcoclaurine 3'-hydroxylase isozyme 2 (probable (S)-N-methylcoclau...,Drought,-1.673854,-1.673854,2.188026e-02


In [12]:
unique_genes = (
    candidate_table["Gene.Names"]
    .str.split()
    .explode()
    .drop_duplicates()
)

print("Unique gene identifiers:")
print(len(unique_genes))

display(unique_genes)

Unique gene identifiers:
16


0       LOC104590414
0       HUJ06_019302
14               NCS
2               OMT4
3       LOC104595105
3            CYP80B2
5                NaN
15      D0Y65_033085
15    glysoja_035139
6               OMT2
7           HNMC4OMT
19      LOC109114309
19      HUJ06_019301
8               OMT6
20            NC6OMT
10      LOC104585771
Name: Gene.Names, dtype: str

In [13]:
gene_level.to_csv(
    RESULT_DIR / "pathway_gene_level_candidates.csv",
    index=False
)

print(
    "Saved:",
    RESULT_DIR / "pathway_gene_level_candidates.csv"
)

Saved: C:\Lumiere\Project\results\pathway_gene_level_candidates.csv


In [14]:
tf_file = SUPP_DIR / "Supplementary Table S7.XLSX"

nac_drought = pd.read_excel(
    tf_file,
    sheet_name="NAC"
)

print("Shape:", nac_drought.shape)

print("\nColumns:")
print(list(nac_drought.columns))

display(nac_drought.head())

Shape: (7, 18)

Columns:
['Entry', 'protein_id', 'Gene_ID', 'baseMean', 'log2FoldChange', 'lfcSE', 'pvalue', 'padj', 'pident', 'evalue', 'bitscore', 'Reviewed', 'Protein.names', 'Gene.Names', 'Organism', 'Gene.Ontology.IDs', 'Protein.families', 'KEGG']


,Entry,protein_id,Gene_ID,baseMean,log2FoldChange,lfcSE,pvalue,padj,pident,evalue,bitscore,Reviewed,Protein.names,Gene.Names,Organism,Gene.Ontology.IDs,Protein.families,KEGG
0,A0A068UGV5,KAF9612150.1,IFM89_038338,153.149089,12.851782,3.241793,4.840000e-12,8.520000e-10,47.6,8.810000e-83,263,unreviewed,NAC domain-containing protein,GSCOC_T00023907001,Coffea canephora (Robusta coffee),GO:0003677; GO:0006355; GO:0016020,NaN,NaN
1,A0A061GJ70,KAF9616689.1,IFM89_031553,122.858368,3.603251,1.013605,2.410000e-05,5.268900e-04,69.8,1.350000e-147,424,unreviewed,NAC domain containing protein 38,TCM_036791,Theobroma cacao (Cacao) (Cocoa),GO:0003677; GO:0006355,NaN,NaN
2,A0A061E1M1,KAF9599729.1,IFM89_001675,418.645388,3.506513,1.927974,1.162295e-03,1.282921e-02,68.4,2.560000e-91,271,unreviewed,"NAC domain protein, IPR003441 isoform 1",TCM_007543,Theobroma cacao (Cacao) (Cocoa),GO:0003677; GO:0006355; GO:0048731,NaN,tcc:18608343;
3,A0A061DNM9,KAF9605666.1,IFM89_018009,23.697951,3.073003,2.037104,3.736397e-03,3.160446e-02,57.0,2.740000e-104,311,unreviewed,"NAC domain protein, IPR003441",TCM_003998,Theobroma cacao (Cacao) (Cocoa),GO:0043068; GO:0043565; GO:0045893; GO:0080187,NaN,tcc:18613123;
4,A0A0M4FLN5,KAF9612417.1,IFM89_039450,226.065415,-1.874359,0.724007,1.460700e-03,1.536445e-02,36.0,1.300000e-33,137,unreviewed,NAC transcription factors 12,MANES_05G163200,Manihot esculenta (Cassava) (Jatropha manihot),GO:0003677; GO:0006355,NaN,NaN


In [15]:
tf_file_drought = SUPP_DIR / "Supplementary Table S7.XLSX"

tf_families = [
    "NAC",
    "MYB",
    "bZIP",
    "WRKY",
    "AP2",
    "ERF",
    "bHLH",
    "HSF",
    "GATA",
    "TCP",
    "GRAS",
    "MADS",
    "ARF",
    "GeBP",
    "HD-ZIP",
    "Aux_IAA",
    "CAMTA",
    "YABBY",
    "Whirly",
    "Zinc finger"
]


drought_tf_list = []

for family in tf_families:
    
    try:
        df = pd.read_excel(
            tf_file_drought,
            sheet_name=family
        )
        
        df["TF_family"] = family
        df["Condition"] = "Drought"
        
        drought_tf_list.append(df)
        
        print(
            family,
            df.shape
        )
        
    except Exception as e:
        print(
            "Skipped:",
            family,
            "|",
            e
        )


drought_tfs = pd.concat(
    drought_tf_list,
    ignore_index=True
)

print("\nCombined TF table:")
print(drought_tfs.shape)

NAC (7, 20)
MYB (16, 20)
bZIP (5, 20)
WRKY (4, 20)
AP2 (11, 20)
ERF (10, 20)
bHLH (6, 20)
HSF (5, 20)
GATA (4, 20)
TCP (1, 20)
GRAS (6, 20)
MADS (3, 20)
ARF (3, 20)
GeBP (2, 20)
HD-ZIP (5, 20)
Aux_IAA (8, 20)
CAMTA (3, 20)
YABBY (2, 20)
Whirly (1, 20)
Zinc finger (16, 20)

Combined TF table:
(118, 20)


In [16]:
print(drought_tfs.columns.tolist())

display(
    drought_tfs[
        [
            "TF_family",
            "Gene.Names",
            "Protein.names",
            "log2FoldChange",
            "padj"
        ]
    ].head(20)
)

['Entry', 'protein_id', 'Gene_ID', 'baseMean', 'log2FoldChange', 'lfcSE', 'pvalue', 'padj', 'pident', 'evalue', 'bitscore', 'Reviewed', 'Protein.names', 'Gene.Names', 'Organism', 'Gene.Ontology.IDs', 'Protein.families', 'KEGG', 'TF_family', 'Condition']


,TF_family,Gene.Names,Protein.names,log2FoldChange,padj
0,NAC,GSCOC_T00023907001,NAC domain-containing protein,12.851782,8.520000e-10
1,NAC,TCM_036791,NAC domain containing protein 38,3.603251,5.268900e-04
2,NAC,TCM_007543,"NAC domain protein, IPR003441 isoform 1",3.506513,1.282921e-02
3,NAC,TCM_003998,"NAC domain protein, IPR003441",3.073003,3.160446e-02
4,NAC,MANES_05G163200,NAC transcription factors 12,-1.874359,1.536445e-02
5,NAC,EUGRSUZ_H00618,Nascent polypeptide-associated complex subunit beta,-2.083084,3.600850e-02
6,NAC,CISIN_1g023427mg,NAC domain-containing protein,-2.418878,4.439005e-03
7,MYB,TCM_021037,Myb domain protein 79 isoform 2,5.998112,4.560000e-05
8,MYB,TCM_022255,Myb domain protein 36,4.638177,2.190000e-06
9,MYB,JCGZ_04192,MYB family protein,3.509408,2.211552e-03


In [17]:
tf_family_counts = (
    drought_tfs["TF_family"]
    .value_counts()
    .reset_index()
)

tf_family_counts.columns = [
    "TF_family",
    "Count"
]

display(tf_family_counts)

,TF_family,Count
0,MYB,16
1,Zinc finger,16
2,AP2,11
3,ERF,10
4,Aux_IAA,8
5,NAC,7
6,bHLH,6
7,GRAS,6
8,bZIP,5
9,HSF,5


In [18]:
top_tfs = drought_tfs.sort_values(
    by="log2FoldChange",
    ascending=False
)

display(
    top_tfs[
        [
            "TF_family",
            "Gene.Names",
            "Protein.names",
            "log2FoldChange",
            "padj"
        ]
    ].head(20)
)

,TF_family,Gene.Names,Protein.names,log2FoldChange,padj
0,NAC,GSCOC_T00023907001,NAC domain-containing protein,12.851782,8.520000e-10
102,Zinc finger,CCACVL1_13427,"Zinc finger, RING-type",9.068711,5.679790e-04
59,HSF,JCGZ_21433,HSF-type DNA-binding domain-containing protein,8.514855,3.260000e-11
75,MADS,LOC104599898 HUJ06_014373,agamous-like MADS-box protein AGL80,8.289291,3.566374e-03
96,CAMTA,100798875 GLYMA_05G178200,CG-1 domain-containing protein,7.213590,4.011330e-04
60,HSF,NaN,Transcription factor HsfB3a,6.418019,2.330000e-05
103,Zinc finger,Ccrd_020437,"Zinc finger, RING/FYVE/PHD-type",6.156196,5.610000e-05
7,MYB,TCM_021037,Myb domain protein 79 isoform 2,5.998112,4.560000e-05
61,HSF,NaN,Transcription factor HsfB3a,5.886187,7.280000e-05
23,bZIP,POPTR_005G126000,BZIP domain-containing protein,5.156454,5.150000e-07


In [19]:
tf_file_heat = SUPP_DIR / "Supplementary Table S8.XLSX"

heat_tf_list = []

for family in tf_families:
    
    try:
        df = pd.read_excel(
            tf_file_heat,
            sheet_name=family
        )
        
        df["TF_family"] = family
        df["Condition"] = "Heat"
        
        heat_tf_list.append(df)
        
        print(
            family,
            df.shape
        )
        
    except Exception as e:
        print(
            "Skipped:",
            family,
            "|",
            e
        )


heat_tfs = pd.concat(
    heat_tf_list,
    ignore_index=True
)

print("\nHeat TF table:")
print(heat_tfs.shape)

NAC (5, 22)
MYB (19, 20)
bZIP (5, 20)
WRKY (6, 20)
AP2 (14, 20)
ERF (13, 21)
bHLH (10, 20)
HSF (8, 20)
GATA (6, 20)
TCP (4, 20)
GRAS (10, 20)
Skipped: MADS | Worksheet named 'MADS' not found
ARF (4, 20)
Skipped: GeBP | Worksheet named 'GeBP' not found
HD-ZIP (5, 22)
Skipped: Aux_IAA | Worksheet named 'Aux_IAA' not found
Skipped: CAMTA | Worksheet named 'CAMTA' not found
YABBY (3, 22)
Whirly (1, 22)
Zinc finger (20, 22)

Heat TF table:
(133, 22)


In [20]:
print("Drought TFs:", drought_tfs.shape)
print("Heat TFs:", heat_tfs.shape)

print("\nTop heat TFs:")
display(
    heat_tfs.sort_values(
        "log2FoldChange",
        ascending=False
    )[
        [
            "TF_family",
            "Gene.Names",
            "Protein.names",
            "log2FoldChange",
            "padj"
        ]
    ].head(10)
)

Drought TFs: (118, 20)
Heat TFs: (133, 22)

Top heat TFs:


,TF_family,Gene.Names,Protein.names,log2FoldChange,padj
35,AP2,DREB1,Dehydration-responsive element-binding protein 1,9.105812,8.950000e-05
49,ERF,DREB1,Dehydration-responsive element-binding protein 1,9.105812,8.950000e-05
5,MYB,TCM_021037,Myb domain protein 79 isoform 2,5.643161,3.726420e-04
113,Zinc finger,BVC80_1821g95,Zinc finger protein,4.330242,4.170000e-08
24,bZIP,BZIP11_2 A4A49_34229,Bzip transcription factor 11,4.216375,4.950010e-04
0,NAC,GSCOC_T00023907001,NAC domain-containing protein,4.192960,7.492670e-04
6,MYB,TCM_009642,"Myb domain protein 62, putative",4.122979,5.540000e-06
7,MYB,MYB11,R2R3-MYB protein,3.953657,2.985290e-04
72,HSF,JCGZ_21433,HSF-type DNA-binding domain-containing protein,3.570457,1.076366e-03
36,AP2,TCM_017321,AP2/B3-like transcriptional factor family protein isoform 1,3.438685,7.450403e-03


In [21]:
tf_file_comparison = SUPP_DIR / "Supplementary Table S9.XLSX"

comparison_tf_list = []

for family in tf_families:
    
    try:
        df = pd.read_excel(
            tf_file_comparison,
            sheet_name=family
        )
        
        df["TF_family"] = family
        df["Condition"] = "Drought_vs_Heat"
        
        comparison_tf_list.append(df)
        
        print(
            family,
            df.shape
        )
        
    except Exception as e:
        print(
            "Skipped:",
            family,
            "|",
            e
        )


comparison_tfs = pd.concat(
    comparison_tf_list,
    ignore_index=True
)

print("\nComparison TF table:")
print(comparison_tfs.shape)

NAC (5, 22)
MYB (19, 20)
bZIP (5, 20)
WRKY (6, 20)
AP2 (14, 20)
ERF (13, 21)
bHLH (10, 20)
HSF (8, 20)
GATA (6, 20)
TCP (4, 20)
GRAS (10, 20)
Skipped: MADS | Worksheet named 'MADS' not found
ARF (4, 20)
Skipped: GeBP | Worksheet named 'GeBP' not found
HD-ZIP (5, 22)
Skipped: Aux_IAA | Worksheet named 'Aux_IAA' not found
Skipped: CAMTA | Worksheet named 'CAMTA' not found
YABBY (3, 22)
Whirly (1, 22)
Zinc finger (20, 22)

Comparison TF table:
(133, 22)


In [22]:
all_tfs = pd.concat(
    [
        drought_tfs,
        heat_tfs,
        comparison_tfs
    ],
    ignore_index=True
)

print(all_tfs.shape)

display(
    all_tfs[
        [
            "Condition",
            "TF_family",
            "Gene.Names",
            "Protein.names",
            "log2FoldChange",
            "padj"
        ]
    ].head()
)

(384, 23)


,Condition,TF_family,Gene.Names,Protein.names,log2FoldChange,padj
0,Drought,NAC,GSCOC_T00023907001,NAC domain-containing protein,12.851782,8.520000e-10
1,Drought,NAC,TCM_036791,NAC domain containing protein 38,3.603251,5.268900e-04
2,Drought,NAC,TCM_007543,"NAC domain protein, IPR003441 isoform 1",3.506513,1.282921e-02
3,Drought,NAC,TCM_003998,"NAC domain protein, IPR003441",3.073003,3.160446e-02
4,Drought,NAC,MANES_05G163200,NAC transcription factors 12,-1.874359,1.536445e-02


In [23]:
tf_master = all_tfs[
    [
        "Condition",
        "TF_family",
        "Gene.Names",
        "Protein.names",
        "log2FoldChange",
        "padj"
    ]
].copy()


# remove empty gene names

tf_master = tf_master[
    tf_master["Gene.Names"].notna()
]


# remove duplicates

tf_master = tf_master.drop_duplicates()


print("Clean TF table:")
print(tf_master.shape)

display(tf_master.head(20))

Clean TF table:
(352, 6)


,Condition,TF_family,Gene.Names,Protein.names,log2FoldChange,padj
0,Drought,NAC,GSCOC_T00023907001,NAC domain-containing protein,12.851782,8.520000e-10
1,Drought,NAC,TCM_036791,NAC domain containing protein 38,3.603251,5.268900e-04
2,Drought,NAC,TCM_007543,"NAC domain protein, IPR003441 isoform 1",3.506513,1.282921e-02
3,Drought,NAC,TCM_003998,"NAC domain protein, IPR003441",3.073003,3.160446e-02
4,Drought,NAC,MANES_05G163200,NAC transcription factors 12,-1.874359,1.536445e-02
5,Drought,NAC,EUGRSUZ_H00618,Nascent polypeptide-associated complex subunit beta,-2.083084,3.600850e-02
6,Drought,NAC,CISIN_1g023427mg,NAC domain-containing protein,-2.418878,4.439005e-03
7,Drought,MYB,TCM_021037,Myb domain protein 79 isoform 2,5.998112,4.560000e-05
8,Drought,MYB,TCM_022255,Myb domain protein 36,4.638177,2.190000e-06
9,Drought,MYB,JCGZ_04192,MYB family protein,3.509408,2.211552e-03


In [24]:
tf_family_condition = (
    tf_master
    .groupby(
        [
            "Condition",
            "TF_family"
        ]
    )
    .size()
    .reset_index(name="Count")
)


display(tf_family_condition)

,Condition,TF_family,Count
0,Drought,AP2,9
1,Drought,ARF,3
2,Drought,Aux_IAA,8
3,Drought,CAMTA,3
4,Drought,ERF,8
5,Drought,GATA,4
6,Drought,GRAS,5
7,Drought,GeBP,2
8,Drought,HD-ZIP,5
9,Drought,HSF,3


In [25]:
for condition in tf_master["Condition"].unique():

    print("\n====================")
    print(condition)

    display(
        tf_master[
            tf_master["Condition"] == condition
        ]
        .sort_values(
            "log2FoldChange",
            ascending=False
        )
        .head(10)
    )


Drought


,Condition,TF_family,Gene.Names,Protein.names,log2FoldChange,padj
0,Drought,NAC,GSCOC_T00023907001,NAC domain-containing protein,12.851782,8.520000e-10
102,Drought,Zinc finger,CCACVL1_13427,"Zinc finger, RING-type",9.068711,5.679790e-04
59,Drought,HSF,JCGZ_21433,HSF-type DNA-binding domain-containing protein,8.514855,3.260000e-11
75,Drought,MADS,LOC104599898 HUJ06_014373,agamous-like MADS-box protein AGL80,8.289291,3.566374e-03
96,Drought,CAMTA,100798875 GLYMA_05G178200,CG-1 domain-containing protein,7.213590,4.011330e-04
103,Drought,Zinc finger,Ccrd_020437,"Zinc finger, RING/FYVE/PHD-type",6.156196,5.610000e-05
7,Drought,MYB,TCM_021037,Myb domain protein 79 isoform 2,5.998112,4.560000e-05
23,Drought,bZIP,POPTR_005G126000,BZIP domain-containing protein,5.156454,5.150000e-07
8,Drought,MYB,TCM_022255,Myb domain protein 36,4.638177,2.190000e-06
104,Drought,Zinc finger,BVC80_7053g2,Zinc finger protein,4.619056,3.060000e-05



Heat


,Condition,TF_family,Gene.Names,Protein.names,log2FoldChange,padj
153,Heat,AP2,DREB1,Dehydration-responsive element-binding protein 1,9.105812,8.950000e-05
167,Heat,ERF,DREB1,Dehydration-responsive element-binding protein 1,9.105812,8.950000e-05
123,Heat,MYB,TCM_021037,Myb domain protein 79 isoform 2,5.643161,3.726420e-04
231,Heat,Zinc finger,BVC80_1821g95,Zinc finger protein,4.330242,4.170000e-08
142,Heat,bZIP,BZIP11_2 A4A49_34229,Bzip transcription factor 11,4.216375,4.950010e-04
118,Heat,NAC,GSCOC_T00023907001,NAC domain-containing protein,4.192960,7.492670e-04
124,Heat,MYB,TCM_009642,"Myb domain protein 62, putative",4.122979,5.540000e-06
125,Heat,MYB,MYB11,R2R3-MYB protein,3.953657,2.985290e-04
190,Heat,HSF,JCGZ_21433,HSF-type DNA-binding domain-containing protein,3.570457,1.076366e-03
154,Heat,AP2,TCM_017321,AP2/B3-like transcriptional factor family protein isoform 1,3.438685,7.450403e-03



Drought_vs_Heat


,Condition,TF_family,Gene.Names,Protein.names,log2FoldChange,padj
286,Drought_vs_Heat,AP2,DREB1,Dehydration-responsive element-binding protein 1,9.105812,8.950000e-05
300,Drought_vs_Heat,ERF,DREB1,Dehydration-responsive element-binding protein 1,9.105812,8.950000e-05
256,Drought_vs_Heat,MYB,TCM_021037,Myb domain protein 79 isoform 2,5.643161,3.726420e-04
364,Drought_vs_Heat,Zinc finger,BVC80_1821g95,Zinc finger protein,4.330242,4.170000e-08
275,Drought_vs_Heat,bZIP,BZIP11_2 A4A49_34229,Bzip transcription factor 11,4.216375,4.950010e-04
251,Drought_vs_Heat,NAC,GSCOC_T00023907001,NAC domain-containing protein,4.192960,7.492670e-04
257,Drought_vs_Heat,MYB,TCM_009642,"Myb domain protein 62, putative",4.122979,5.540000e-06
258,Drought_vs_Heat,MYB,MYB11,R2R3-MYB protein,3.953657,2.985290e-04
323,Drought_vs_Heat,HSF,JCGZ_21433,HSF-type DNA-binding domain-containing protein,3.570457,1.076366e-03
287,Drought_vs_Heat,AP2,TCM_017321,AP2/B3-like transcriptional factor family protein isoform 1,3.438685,7.450403e-03


In [26]:
tf_master.to_csv(
    RESULT_DIR / "tf_master.csv",
    index=False
)

print("TF table saved")

TF table saved


In [28]:
from pathlib import Path
import pandas as pd

RESULT_DIR = Path("D:/Lumiere/Project/results")

# Reload your pathway candidate table
pathway_genes = pd.read_csv(
    RESULT_DIR / "pathway_gene_level_candidates.csv"
)

print("Loaded pathway genes:")
print(pathway_genes.shape)

display(pathway_genes.head())

Loaded pathway genes:
(18, 6)


,Gene.Names,Protein.names,Condition,max_log2FC,min_log2FC,best_padj
0,NaN,Berberine bridge enzyme (EC 1.21.3.3),Drought,-2.015363,-2.015363,0.001470
1,NaN,Berberine bridge enzyme (EC 1.21.3.3),Drought_vs_Heat,-2.511092,-2.511092,0.000022
2,NaN,Stylopine synthase,Drought_vs_Heat,-4.279584,-4.279584,0.000003
3,NaN,Stylopine synthase,Heat,-2.247584,-2.247584,0.019634
4,D0Y65_033085 glysoja_035139,S-norcoclaurine synthase (S-norcoclaurine synthase 2),Drought_vs_Heat,-2.245025,-2.860257,0.004516


In [29]:
pathway_genes.to_csv(
    RESULT_DIR / "pathway_genes_clean.csv",
    index=False
)

print("Saved pathway genes")

Saved pathway genes
